# CIFAR-10 CNN -> hls4ml IP core (v0.2 stage 3)

Adapted from ICTP `02-cnn/02.hls4ml/cnn-hls4ml.ipynb`.

## What this notebook does

1. Loads the distilled CIFAR-10 student model trained in stage 2.
2. Builds an `hls4ml` configuration with the same defaults as the ICTP
   reference (`ap_fixed<16,12>` precision, `reuse_factor=8`, Latency strategy,
   `io_stream` IO).
3. Converts the Keras model to an HLS project targeting **`xczu3eg-sfvc784-2-e`**
   (our AUP-ZU3 silicon).
4. Runs C-synthesis through Vitis HLS and exports the IP core.
5. Compares the bit-accurate hls4ml prediction against the QKeras
   software prediction on the CIFAR-10 test set, so we know what to expect
   from the FPGA before it runs.

## What stays the same vs MNIST upstream

- HLS config keys (`Strategy`, `IOType`, `RoundingMode`, `SaturationMode`).
- Precision and reuse_factor defaults.
- The two-step verification (predict / hls compare) pattern.

## What changes vs MNIST upstream

- Input shape: `(32, 32, 3)` instead of `(28, 28, 1)`.
- Model path: `../01.training/models/distilled_cifar10.h5`.
- Output project path: `output/myproject_cifar10_prj`.

In [ ]:
# === Libraries ===
import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import cifar10

import qkeras
import hls4ml

from sklearn.metrics import accuracy_score, confusion_matrix

print('TensorFlow:', tf.__version__)
print('QKeras:', qkeras.__version__)
print('hls4ml:', hls4ml.__version__)

## 1. Path to the Vitis HLS installation

hls4ml shells out to `vitis_hls` for C-synthesis. We point at the
Vivado/Vitis 2024.1 installation already on this machine.


In [ ]:
# Make Vitis HLS visible to subprocesses spawned by hls4ml
VITIS_HLS_DIR = '/tools/amd/2024.1/Vitis_HLS/2024.1'
VITIS_DIR     = '/tools/amd/2024.1/Vitis/2024.1'
VIVADO_DIR    = '/tools/amd/2024.1/Vivado/2024.1'

for d in (VITIS_HLS_DIR, VITIS_DIR, VIVADO_DIR):
    bin_dir = os.path.join(d, 'bin')
    if bin_dir not in os.environ['PATH']:
        os.environ['PATH'] = bin_dir + os.pathsep + os.environ['PATH']

os.environ['XILINX_HLS']    = VITIS_HLS_DIR
os.environ['XILINX_VITIS']  = VITIS_DIR
os.environ['XILINX_VIVADO'] = VIVADO_DIR

# Verify vitis_hls is reachable
import shutil
print('vitis_hls:', shutil.which('vitis_hls'))
print('vivado   :', shutil.which('vivado'))

## 2. Load the distilled student model

QKeras layers need their custom objects registered when loading.


In [ ]:
from qkeras.utils import _add_supported_quantized_objects
co = {}
_add_supported_quantized_objects(co)

model = keras.models.load_model('../01.training/models/distilled_cifar10.h5', custom_objects=co)
model.summary()

## 3. Reload the CIFAR-10 test set

Same preprocessing as in stage 2.


In [ ]:
(_, _), (x_test_raw, y_test_raw) = cifar10.load_data()
x_test = x_test_raw.astype('float32') / 255.0
y_test_int = y_test_raw.flatten()

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print('Test set:', x_test.shape, y_test_int.shape)

# Software (QKeras) reference: what the model produces in Python
y_pred_sw = np.argmax(model.predict(x_test, batch_size=128, verbose=0), axis=1)
acc_sw    = accuracy_score(y_test_int, y_pred_sw)
print(f'Student software accuracy (reference): {acc_sw*100:.2f} %')

## 4. Generate the hls4ml configuration

Defaults follow the ICTP MNIST notebook exactly so the comparison stays
controlled.

Knobs we keep at the ICTP defaults:

- `granularity='name'`  -> per-layer override allowed if needed
- `default_precision='fixed<16,12>'` -> 16-bit, 12 integer bits, 4 fractional
- `default_reuse_factor=8` -> DSP shared 8 times across logical multiplications
- `Strategy='Latency'` -> minimise inference time, not area
- `IOType='io_stream'` -> AXI-Stream between layers
- `RoundingMode='AP_RND_CONV'` -> convergent rounding
- `SaturationMode='AP_SAT'` -> clamp on overflow

In [ ]:
hls_config = hls4ml.utils.config_from_keras_model(
    model,
    granularity='name',
    # 16 bits total, 6 integer bits, 10 fractional bits.
    # Previous attempt used fixed<16,12> (ICTP MNIST default) and the
    # hls4ml accuracy collapsed to 10 % (random for 10 classes) because
    # the 4 fractional bits could not represent products smaller than
    # ~0.06. With <16,6> we get ~0.001 resolution and the integer range
    # stays comfortably above what this network needs (activations are
    # bounded to roughly [-4, +4] by QKeras).
    default_precision='fixed<16,6>',
    default_reuse_factor=8,
)

hls_config['Model']['Strategy']       = 'Latency'
hls_config['Model']['IOType']         = 'io_stream'
hls_config['Model']['RoundingMode']   = 'AP_RND_CONV'
hls_config['Model']['SaturationMode'] = 'AP_SAT'

# --- Per-layer overrides (v3 plan) ---
#
# fc1: 1024 -> 32 = 32 768 MACs. We have measured the LUT cost at two
# reuse points: reuse=256 -> 28 685 LUTs, reuse=128 -> 41 697 LUTs.
# Lowering reuse actually *increased* LUT cost because the dense_resource
# kernel adds more multiplexer logic than it saves in DSP usage. Pushing
# the other direction (reuse=512) is expected to keep cutting both LUTs
# and DSPs because the layer is far off the dataflow critical path
# (conv2 dominates at ~78 us), so the extra cycles of fc1 (~6.5 us at
# reuse=512) stay invisible at system level.
hls_config['LayerName']['fc1']['Strategy']    = 'Resource'
hls_config['LayerName']['fc1']['ReuseFactor'] = 512

# conv2: 8 -> 16 channels, 16 x 16 = 18 432 MACs. Measured in the
# previous round at 5 652 LUTs + 32 DSPs with Resource/reuse=16, a 78 %
# LUT reduction vs Latency/reuse=8. Keep this setting.
hls_config['LayerName']['conv2']['Strategy']    = 'Resource'
hls_config['LayerName']['conv2']['ReuseFactor'] = 16

# output: 32 -> 10 = 320 MACs. Small layer but currently consuming
# 7 467 LUTs at 0 DSPs because Latency strategy generates combinational
# multipliers. Pushing to Resource should swap that LUT mass for ~40
# DSPs (which we have plenty of) at negligible latency cost.
hls_config['LayerName']['output']['Strategy']    = 'Resource'
hls_config['LayerName']['output']['ReuseFactor'] = 8

import pprint
pprint.pprint(hls_config)

## 5. Convert the model to an HLS project

Target the AUP-ZU3 silicon explicitly (`xczu3eg-sfvc784-2-e`). The output
lives under `output/myproject_cifar10_prj/`.


In [ ]:
OUTPUT_DIR  = 'output/myproject_cifar10_prj'
PROJECT_NAM = 'myproject'  # default name expected by the AXI wrapper template

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=hls_config,
    output_dir=OUTPUT_DIR,
    project_name=PROJECT_NAM,
    backend='Vitis',
    part='xczu3eg-sfvc784-2-e',
    clock_period=10,  # ns -> 100 MHz
    io_type='io_stream',
)

print('HLS project created at:', OUTPUT_DIR)

In [ ]:
# Plot the parsed model topology to confirm hls4ml understood the layers.
# Requires pydot + graphviz; if not installed, just skip — purely cosmetic.
try:
    hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)
except Exception as e:
    print('plot_model skipped:', e)

## 6. Compile the hls4ml model (Python-level bit-accurate simulator)

`hls_model.compile()` builds a CPython C++ extension implementing the same
fixed-point arithmetic that will run on the FPGA. It lets us verify what the
hardware will produce before paying the cost of full HLS synthesis.

In [ ]:
hls_model.compile()

In [ ]:
# Predict on the full test set with the bit-accurate hls4ml model.
# hls4ml requires a C-contiguous float32 array; .astype() does not always
# guarantee that, so use np.ascontiguousarray to be safe.
x_test_c = np.ascontiguousarray(x_test, dtype=np.float32)
y_pred_hls_probs = hls_model.predict(x_test_c)
y_pred_hls = np.argmax(y_pred_hls_probs, axis=1)
acc_hls = accuracy_score(y_test_int, y_pred_hls)

print(f'Student software (float / QKeras):   {acc_sw*100:.2f} %')
print(f'Student hls4ml  (fixed<16,6>):       {acc_hls*100:.2f} %')
print(f'Quantisation drop:                   {(acc_sw-acc_hls)*100:.2f} pp')

## 7. C-synthesis with Vitis HLS

This is the expensive step: Vitis HLS turns the C++ project into a
synthesisable Verilog IP and produces a resource and latency report. On
an i7 CPU expect anywhere between 10 and 30 minutes.

When it finishes, the IP is exported into
`output/myproject_cifar10_prj/myproject_prj/solution1/impl/ip/` ready for
Vivado to pick up as an IP repository.

In [ ]:
hls_model.build(
    csim=False,
    synth=True,
    cosim=False,
    validation=False,
    export=True,
    vsynth=False,
)

In [ ]:
# Resource and latency report after synthesis
hls4ml.report.read_vivado_report(OUTPUT_DIR)

## Outputs and what to do next

- IP core: `output/myproject_cifar10_prj/myproject_prj/solution1/impl/ip/`
- Reports : same location, sub-folders `report/` and `syn/report/`.
- Software acc / hls4ml acc gap recorded above.

The next stage (`03.hls/`) wraps this IP with an AXI-Stream + AXI-Lite
shell so a DMA in the PL can drive it. Stage `04.hw/` then drops the IP
into a Vivado block design and produces the bitstream.